# 03 · CRUD en Cassandra

Keyspace `github_metrics`. Driver: `cassandra-driver` (CQL).

**Justificación:** escrituras masivas append-only y lecturas por rango de tiempo; cada tabla se diseña para una consulta.

In [ ]:
import sys, os
sys.path.insert(0, "../src")          # módulos del proyecto (config, extract, clean, ...)
import config
from datetime import datetime, timezone, date
from pprint import pprint

In [ ]:
s = config.cassandra_session()
for r in s.execute("SELECT table_name FROM system_schema.tables WHERE keyspace_name='github_metrics'"): print(r.table_name)

## C · Insertar

In [ ]:
demo_day = date(2099, 1, 1)   # día ficticio para no mezclar con datos reales
s.execute('INSERT INTO events_by_hour (event_date, event_hour, event_type, event_count) VALUES (%s,%s,%s,%s)',
          (demo_day, 10, 'PushEvent', 123))
s.execute('INSERT INTO events_by_hour (event_date, event_hour, event_type, event_count) VALUES (%s,%s,%s,%s)',
          (demo_day, 11, 'PushEvent', 456))
print('insertado')

## R · Consultar

In [ ]:
rows = s.execute('SELECT * FROM events_by_hour WHERE event_date=%s', (demo_day,))
for r in rows: print(r)

In [ ]:
# Consulta real: eventos por hora del último día cargado
last = s.execute('SELECT event_date FROM daily_totals WHERE year=%s LIMIT 1', (date.today().year,)).one()
if last:
    d = last.event_date.date()
    print('día:', d)
    for r in s.execute('SELECT event_hour, event_type, event_count FROM events_by_hour WHERE event_date=%s LIMIT 10', (d,)): print(r)

In [ ]:
# KPIs calculados por el DAG
if last:
    for r in s.execute('SELECT kpi_name, kpi_value, kpi_label FROM kpis_by_date WHERE event_date=%s', (d,)): print(r)

## U · Actualizar

In [ ]:
s.execute('UPDATE events_by_hour SET event_count=%s WHERE event_date=%s AND event_hour=%s AND event_type=%s',
          (999, demo_day, 10, 'PushEvent'))
print(s.execute('SELECT * FROM events_by_hour WHERE event_date=%s AND event_hour=%s', (demo_day, 10)).one())

## D · Eliminar

In [ ]:
s.execute('DELETE FROM events_by_hour WHERE event_date=%s AND event_hour=%s AND event_type=%s', (demo_day, 10, 'PushEvent'))
s.execute('DELETE FROM events_by_hour WHERE event_date=%s', (demo_day,))   # borra la partición completa
print('quedan:', list(s.execute('SELECT * FROM events_by_hour WHERE event_date=%s', (demo_day,))))